# LM Consistency Benchmark — Analysis

**Research Question:** Do frontier LLMs produce the same answer when a question is rephrased? Does consistency vary by domain, and does it correlate with accuracy?

This notebook analyzes results from two benchmark tracks:
- **MMLU Track:** Paraphrase consistency across multiple-choice questions (57 subjects)
- **PAWS Track:** Order-sensitivity in paraphrase detection

Model evaluated: `claude-haiku-4-5-20251001`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path

# Configure plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

RESULTS_DIR = Path("../results")
FIGURES_DIR = Path("../figures")

MODEL = "claude-haiku-4-5-20251001"

## MMLU Results

In [ ]:
# Load MMLU report
mmlu_report_path = RESULTS_DIR / f"mmlu_report_{MODEL}.json"
with open(mmlu_report_path) as f:
    mmlu = json.load(f)

print("=" * 50)
print("MMLU Track — Key Metrics")
print("=" * 50)
print(f"Model:                  {mmlu['model']}")
print(f"Questions evaluated:    {mmlu['n_questions']}")
print(f"Avg variants/question:  {mmlu['n_variants_per_question']:.2f}")
print()
print(f"Consistency Rate:       {mmlu['overall_consistency_rate']:.1%}")
print(f"Overall Accuracy:       {mmlu['overall_accuracy']:.1%}")
print(f"Krippendorff's α:       {mmlu['krippendorff_alpha']:.3f}")
print()
print(f"Accuracy (original):    {mmlu['by_paraphrase_type']['accuracy']['original']:.1%}  "
      f"(n={mmlu['by_paraphrase_type']['count']['original']})")
print(f"Accuracy (paraphrase):  {mmlu['by_paraphrase_type']['accuracy']['paraphrase']:.1%}  "
      f"(n={mmlu['by_paraphrase_type']['count']['paraphrase']})")
print()
print("Pearson r (acc vs consistency): "
      f"{mmlu['acc_consistency_pearson_r']:.3f}  "
      f"(p={mmlu['acc_consistency_pearson_p']:.3f})")
print()
print("-" * 50)
print("By Category:")
print("-" * 50)
cat_df = pd.DataFrame(mmlu["by_category"])
cat_df.columns = ["Category", "Consistency Rate", "Accuracy", "n"]
cat_df["Consistency Rate"] = cat_df["Consistency Rate"].map("{:.1%}".format)
cat_df["Accuracy"] = cat_df["Accuracy"].map("{:.1%}".format)
print(cat_df.to_string(index=False))

In [ ]:
# Display MMLU figures
mmlu_figures = [
    (f"consistency_by_category_{MODEL}.png", "Consistency Rate by Category"),
    (f"acc_vs_consistency_{MODEL}.png", "Accuracy vs. Consistency"),
    (f"orig_vs_para_accuracy_{MODEL}.png", "Original vs. Paraphrase Accuracy"),
    (f"answer_heatmap_{MODEL}.png", "Answer Choice Heatmap"),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for ax, (fname, title) in zip(axes, mmlu_figures):
    img_path = FIGURES_DIR / fname
    if img_path.exists():
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(title, fontsize=12, fontweight="bold", pad=8)
    else:
        ax.text(0.5, 0.5, f"Figure not found:\n{fname}",
                ha="center", va="center", transform=ax.transAxes, color="gray")
    ax.axis("off")

plt.suptitle("MMLU Track — Figures", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## PAWS Results

In [ ]:
# Load PAWS report
paws_report_path = RESULTS_DIR / f"paws_report_{MODEL}.json"
with open(paws_report_path) as f:
    paws = json.load(f)

print("=" * 50)
print("PAWS Track — Key Metrics")
print("=" * 50)
print(f"Model:                      {paws['model']}")
print()
print(f"Flip Rate (overall):        {paws['overall_flip_rate']:.1%}")
print(f"Flip Rate (true para.):     {paws['flip_rate_true_paraphrase']:.1%}")
print(f"Flip Rate (non-para.):      {paws['flip_rate_non_paraphrase']:.1%}")
print()
print(f"Forward Accuracy:           {paws['fwd_accuracy']:.1%}")
print(f"Reverse Accuracy:           {paws['rev_accuracy']:.1%}")
print(f"Mean Accuracy:              {paws['mean_accuracy']:.1%}")

In [ ]:
# Display PAWS figure
paws_fig_path = FIGURES_DIR / f"paws_flip_rate_{MODEL}.png"

fig, ax = plt.subplots(figsize=(9, 5))
if paws_fig_path.exists():
    img = mpimg.imread(str(paws_fig_path))
    ax.imshow(img)
    ax.set_title("PAWS Flip Rate by Pair Type", fontsize=13, fontweight="bold", pad=8)
else:
    ax.text(0.5, 0.5, "Figure not found", ha="center", va="center",
            transform=ax.transAxes, color="gray")
ax.axis("off")
plt.tight_layout()
plt.show()

## Key Takeaways

- **Consistency is only moderate (~52.6% on MMLU):** The model gives the same answer across all paraphrase variants for barely half of questions, meaning surface-level phrasing changes cause answer flips nearly as often as not.

- **Krippendorff's α of 0.509 indicates unreliable agreement:** This falls far below the 0.8 threshold typically required for reliable annotation, confirming that model outputs are substantially influenced by prompt wording rather than pure reasoning.

- **Category matters — Humanities show the worst consistency (25%):** STEM and Professional/Applied questions are more stable, while open-ended Humanities questions appear most susceptible to paraphrase-induced answer changes.

- **PAWS flip rates reveal a meaningful asymmetry:** The model flips its judgment on 25.6% of true paraphrase pairs but 66.7% of adversarial non-paraphrase pairs. This asymmetry shows the model does pick up on genuine semantic differences, but is still highly sensitive to superficial lexical variation.

- **Accuracy alone understates reliability risk:** The model achieves 36.4% accuracy on MMLU and 58.0% on PAWS, but accuracy metrics mask the fact that correct answers may not be stable — a model could be right for the wrong reasons, and rephrase the same question to get a different (also wrong) answer.

In [ ]:
# Load raw MMLU results and compute per-subject consistency
mmlu_jsonl_path = RESULTS_DIR / f"mmlu_{MODEL}_n100.jsonl"

records = []
with open(mmlu_jsonl_path) as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.DataFrame(records)
print(f"Loaded {len(df)} rows across {df['subject'].nunique()} subjects")
df.head()

In [ ]:
# Compute per-subject consistency rate
# A question is consistent if all its paraphrase variants have the same predicted answer
question_consistency = (
    df.groupby(["subject", "original_idx"])["predicted"]
    .agg(lambda preds: preds.nunique() == 1)
    .reset_index()
    .rename(columns={"predicted": "consistent"})
)

subject_stats = (
    question_consistency
    .groupby("subject")["consistent"]
    .agg(consistency_rate="mean", n_questions="count")
    .reset_index()
    .sort_values("consistency_rate", ascending=True)
)

# Bar chart
fig, ax = plt.subplots(figsize=(12, max(5, len(subject_stats) * 0.35)))

colors = [
    "#d62728" if r < 0.4 else "#ff7f0e" if r < 0.6 else "#2ca02c"
    for r in subject_stats["consistency_rate"]
]

bars = ax.barh(
    subject_stats["subject"].str.replace("_", " ").str.title(),
    subject_stats["consistency_rate"],
    color=colors,
    edgecolor="white",
    linewidth=0.5,
)

# Annotate bar values
for bar, rate, n in zip(bars, subject_stats["consistency_rate"], subject_stats["n_questions"]):
    ax.text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{rate:.0%}  (n={n})",
        va="center", ha="left", fontsize=8, color="#333333",
    )

ax.axvline(subject_stats["consistency_rate"].mean(), color="steelblue",
           linestyle="--", linewidth=1.2, label="Mean consistency")

ax.set_xlim(0, 1.25)
ax.set_xlabel("Consistency Rate", fontsize=11)
ax.set_title(
    f"Per-Subject Consistency Rate — {MODEL}",
    fontsize=13, fontweight="bold", pad=10,
)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
sns.despine(left=True)
plt.tight_layout()
plt.show()

print("\nSubjects with lowest consistency:")
print(subject_stats.head(5).to_string(index=False))
print("\nSubjects with highest consistency:")
print(subject_stats.tail(5).to_string(index=False))